# A. Catalog Crossmatch - 03/10/2026

This notebook queries the databases of Rubin Observatory Data Preview 1 (DP1) and Euclid Early Release data to find and analyze galaxies within a selected sky region.

Features include:
A) Query DP1 objects within a region. B) Obtain redshift estimates of such objects using zRail. C) Query Euclid objects in that same region.  D) Crossmatch these sources based on coordinates. E) Compare object fluxes in different apertures. Sersic best fit and will be the primary filter used from Euclid. F) Plot the flux of each filter in its respective wavelength for an individual galaxies. Spectral Energy Distribution. 

Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import getpass

from lsst.rsp import get_tap_service, retrieve_query
from lsst.daf.butler import Butler, CollectionType
from astroquery.utils.tap.core import TapPlus
from IPython.display import clear_output
from astropy.table import Table, join
from astropy.coordinates import SkyCoord
from astropy import units as u
from pathlib import Path

If you receive a file "KB_462" no need to change, Proceed

In [ ]:
my_username = getpass.getuser()

## IF YOU CHANGE/ WORKING UNDER DIFFERENT FOLDER YOU MUST CHANGE THIS , KB_462 IS MINE
BASE = Path("/home/"+my_username+"/KB_462")
DATA = BASE / "data"

print("Working dir:", Path().resolve())
print(BASE)

## Part 1

# LSST Query 1/2

Query Rubin DP1 for: Object ID, ra, dec, u,g,r,i,z,y, sersic, sersic errors, and more. Builds Astrophy table of Rubin DP1 LSST objects.

Query keyword must be added in a few spots, surveyLSST (1/2) and crossmatchCats (2/2)

In [ ]:
# LSST QUERY

def surveyLSST(center_ra,center_dec,radius):
    
    # UPDATE BUTLER FOR FUTURE RELEASES
    butler = Butler("dp1", collections="LSSTComCam/DP1")
    #
    
    tap = get_tap_service('tap')

    # ADD LSST QUERY KEYWORD IN MULTIPLE PLACES, surveyLSST AND crossmatchCats
    query = f"""SELECT coord_ra, coord_dec, g_extendedness, tract, patch, sersic_index, sersic_x, sersic_y, sersic_reff_x, sersic_reff_y, sersic_rho, sersic_ra, sersic_dec, u_sersicFlux, g_sersicFlux, r_sersicFlux, i_sersicFlux, z_sersicFlux,y_sersicFlux,
                       u_sersicFluxErr, g_sersicFluxErr, r_sersicFluxErr, i_sersicFluxErr, z_sersicFluxErr,y_sersicFluxErr, objectId
            FROM dp1.Object
            WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec),
            CIRCLE('ICRS', {center_ra}, {center_dec}, {radius/60})) = 1"""
    #
    
    job = tap.submit_job(query)
    job.run()
    job.wait(phases=['COMPLETED', 'ERROR'])
    #print('Job phase is', job.phase)
    if job.phase == 'ERROR':
        job.raise_if_error()
    assert job.phase == 'COMPLETED'
    object_results = job.fetch_result()
    
    ra_star = []
    dec_star = []
    ra_gal = []
    dec_gal = []
    
    #LSST KEYWORD HERE
    Sersic_Index = []
    Tract = []
    Patch = []
    Sersic_x = []
    Sersic_y = []
    Sersic_REff_X = []
    Sersic_REff_Y = []
    Sersic_rho = []
    Sersic_ra = []
    Sersic_dec = []
    #
    
    uFlux = []
    gFlux = []
    rFlux = []
    iFlux = []
    zFlux = []
    yFlux = []
    uFluxErr = []
    gFluxErr = []
    rFluxErr = []
    iFluxErr = []
    zFluxErr = []
    yFluxErr = []
    objID = []
     
    for row in object_results:
        if row[f'g_extendedness'] == 0.0:
            ra_star.append(row['coord_ra'])
            dec_star.append(row['coord_dec'])
        if row[f'g_extendedness'] == 1.0:
            ra_gal.append(row['coord_ra'])
            dec_gal.append(row['coord_dec'])
            
            #LSST KEYWORD HERE
            Sersic_Index.append(row['sersic_index'])
            Tract.append(row['tract'])
            Patch.append(row['patch'])
            Sersic_x.append(row['sersic_x'])
            Sersic_y.append(row['sersic_y'])
            Sersic_REff_X.append(row['sersic_reff_x'])
            Sersic_REff_Y.append(row['sersic_reff_y'])
            Sersic_rho.append(row['sersic_rho'])
            Sersic_ra.append(row['sersic_ra'])
            Sersic_dec.append(row['sersic_dec'])
            #
            
            uFlux.append(row['u_sersicFlux'])
            gFlux.append(row['g_sersicFlux'])
            rFlux.append(row['r_sersicFlux'])
            iFlux.append(row['i_sersicFlux'])
            zFlux.append(row['z_sersicFlux'])
            yFlux.append(row['y_sersicFlux'])
            uFluxErr.append(row['u_sersicFluxErr'])
            gFluxErr.append(row['g_sersicFluxErr'])
            rFluxErr.append(row['r_sersicFluxErr'])
            iFluxErr.append(row['i_sersicFluxErr'])
            zFluxErr.append(row['z_sersicFluxErr'])
            yFluxErr.append(row['y_sersicFluxErr'])
            objID.append(row['objectId'])
            
#LSST KEYWORD HERE
    lsst_data = list(zip(ra_gal,dec_gal,Sersic_Index,Tract,Patch,Sersic_x,Sersic_y,Sersic_REff_X,Sersic_REff_Y,Sersic_rho,Sersic_ra,Sersic_dec,uFlux,gFlux,rFlux,iFlux,zFlux,yFlux,uFluxErr,gFluxErr,rFluxErr,iFluxErr,zFluxErr,yFluxErr))
    lsst_table = pd.DataFrame(lsst_data, columns=["ra", "dec","Sersic_Index","Tract","Patch","Sersic_x","Sersic_y","Sersic_REff_X","Sersic_REff_Y","Sersic_rho","Sersic_ra","Sersic_dec","uFlux","gFlux","rFlux","iFlux","zFlux","yFlux","uFluxErr","gFluxErr","rFluxErr","iFluxErr","zFluxErr","yFluxErr"])
    lsst_table
#


    return{
        'ra_lsst': ra_gal,
        'dec_lsst': dec_gal,
        
        #LSST KEYWORD HERE
        'Sersic_Index' : Sersic_Index,
        'Tract': Tract,
        'Patch': Patch,
        'Sersic_x':Sersic_x,
        'Sersic_y':Sersic_y,
        'Sersic_REff_X':Sersic_REff_X,
        'Sersic_REff_Y':Sersic_REff_Y,
        'Sersic_rho':Sersic_rho,
        'Sersic_ra':Sersic_ra,
        'Sersic_dec':Sersic_dec,
        #
        
        'uFlux': uFlux,
        'gFlux': gFlux,
        'rFlux': rFlux,
        'iFlux': iFlux,
        'zFlux': zFlux,
        'yFlux': yFlux,
        'uFluxErr': uFluxErr,
        'gFluxErr': gFluxErr,
        'rFluxErr': rFluxErr,
        'iFluxErr': iFluxErr,
        'zFluxErr': zFluxErr,
        'yFluxErr': yFluxErr,
        'objID': objID
          }

# zRail Redshift Estimate

Use zRail (seperate lsst pipleine RAIL) to get estimated redshift values for DP1 objects, using object_photoz.parquet to filter out foreground and background galaxies outside of the range: z_center +- z_range. 

In [ ]:
# REDSHIFT ESTIMATE, DP1 FEATURE
def zRail(lsst,z_center,z_range):

    dp1_pz_df = pd.read_parquet(

        # UPDATE FOR FUTURE RELEASES
        'https://data.lsdb.io/hats/dp1/object_photoz.parquet',
        #
        
        columns=['objectId', 'lephare_z_mean'],
        storage_options={"anon": True}   )

    rail_ids = dp1_pz_df['objectId'].values
    rail_z   = dp1_pz_df['lephare_z_mean'].values
    
    ra_gal = lsst['ra_lsst']
    dec_gal = lsst['dec_lsst']
    objID = lsst['objID']
    ra_gal = np.asarray(ra_gal)
    dec_gal = np.asarray(dec_gal)
       
    obj_idx = []
    idx = []
    for i in objID:
        ind = (np.where(rail_ids == i)[0])
        if len(ind) == 0:
            continue
        else:
            idx.append(int(ind))
            obj_idx.append(int(np.where(objID == i)[0]))
            
    match = np.asarray(rail_ids[idx])
    match_ids = []

    for i in match: match_ids.append(int(i))
        
    match = np.asarray(rail_z[idx])
    match_z = []
    for i in match: match_z.append(float(i))

    match = (ra_gal[obj_idx])
    match_ra = []
    for i in match: match_ra.append(i)
    
    match = np.asarray(dec_gal[obj_idx])
    match_dec = []
    for i in match: match_dec.append(i)

    gal_data = zip(match_ra,match_dec,match_z)
    print()
    print('Retreiving Redshift Data From RAIL')

    r = []
    d = []
    z = []
    for data in gal_data:
        ras = data[0]
        r.append(ras)
        decs = data[1]
        d.append(decs)
        zs = data[2]
        z.append(zs)

    member_gal_coords = []
    member_ra = []
    member_dec = []
    member_z = []
    for (ra_z, dec_z, redshift) in list(zip(r,d,z)):
        if (z_center - z_range) <= redshift <= (z_center+ z_range):
            member_ra.append(ra_z)
            member_dec.append(dec_z)
            member_z.append(redshift)
    #print(f'Redshift Range: z = [{z_center - z_range} , {z_center + z_range}]')
    print(f'Number of Member Galaxies ', len(member_ra))  

    mem_tag = []
    for i in match_z:
        if z_center - z_range <= i <= z_center + z_range:
            mem_tag.append(1)
        else:
            mem_tag.append(0)
        
    rail_data = list(zip(match_ra,match_dec,match_z,mem_tag,objID))
    rail_table = pd.DataFrame(rail_data, columns=["RA", "DEC","Z","Member Gal","LSST-ID"])
    rail_table
    
    return{
        'objID': objID,
        'member_ra':member_ra,
        'member_dec': member_dec,
        'member_z': member_z,
        'mem_tag': mem_tag,
        'match_z': match_z
          }

# Merge zRail and LSST Query

Expected Error: A) UnitsWarning: The unit 'Angstrom' has been deprecated in the VOUnit standard. Suggested: 0.1nm.

In [ ]:
# CATALOG MERGER
#Finding Catalogs with "mer"

euclid = TapPlus(url="https://eas.esac.esa.int/tap-server/tap")
query = """
SELECT schema_name, table_name
FROM TAP_SCHEMA.tables
WHERE table_name LIKE '%mer%'
"""
job = euclid.launch_job_async(query).get_results()
job

query2 = """
SELECT TOP 10 *
FROM catalogue.mer_catalogue
"""
job2 = euclid.launch_job_async(query2).get_results()
job2

# Euclid Query 1/1

Query Euclid for these some galaxies based on spacial position, for flux of the filters Vis, Y, J, H, as well as flux in apertures of 1, 2, and 3 FWHM, and respective errors. Builds Euclid table.

In [ ]:
# EUCLID QUERY 

def surveyEUCLID(center_ra,center_dec,radius):

    euclid = TapPlus(url="https://eas.esac.esa.int/tap-server/tap")

    #EUCLID KEYWORD HERE
    query = f""" SELECT "right_ascension","declination",
                        "flux_vis_1fwhm_aper","flux_vis_2fwhm_aper","flux_vis_3fwhm_aper",
                        "fluxerr_vis_1fwhm_aper", "fluxerr_vis_2fwhm_aper", "fluxerr_vis_3fwhm_aper",
                        "flux_y_1fwhm_aper","flux_y_2fwhm_aper","flux_y_3fwhm_aper",
                        "fluxerr_y_1fwhm_aper", "fluxerr_y_2fwhm_aper", "fluxerr_y_3fwhm_aper",
                        "flux_j_1fwhm_aper","flux_j_2fwhm_aper","flux_j_3fwhm_aper",
                        "fluxerr_j_1fwhm_aper", "fluxerr_j_2fwhm_aper", "fluxerr_j_3fwhm_aper",
                        "flux_h_1fwhm_aper","flux_h_2fwhm_aper","flux_h_3fwhm_aper",
                        "fluxerr_h_1fwhm_aper", "fluxerr_h_2fwhm_aper", "fluxerr_h_3fwhm_aper",
                        "flux_vis_sersic", "flux_y_sersic", "flux_j_sersic", "flux_h_sersic",
                        "fluxerr_vis_sersic","fluxerr_y_sersic", "fluxerr_j_sersic", "fluxerr_h_sersic"   
                 FROM mer_catalogue
                 WHERE CONTAINS(
                  POINT('ICRS', right_ascension, declination),
                  CIRCLE('ICRS', {center_ra}, {center_dec}, {radius/60})) = 1 """
    #
    
    result = euclid.launch_job_async(query).get_results()
    #result
    
    #EUCLID KEYWORD HERE
    cols = [
        "right_ascension","declination",
        "flux_vis_1fwhm_aper","flux_vis_2fwhm_aper","flux_vis_3fwhm_aper",
        "fluxerr_vis_1fwhm_aper", "fluxerr_vis_2fwhm_aper", "fluxerr_vis_3fwhm_aper",
        
        "flux_y_1fwhm_aper","flux_y_2fwhm_aper","flux_y_3fwhm_aper",
        "fluxerr_y_1fwhm_aper", "fluxerr_y_2fwhm_aper", "fluxerr_y_3fwhm_aper", 
        
        "flux_j_1fwhm_aper","flux_j_2fwhm_aper","flux_j_3fwhm_aper",
        "fluxerr_j_1fwhm_aper", "fluxerr_j_2fwhm_aper", "fluxerr_j_3fwhm_aper", 
        
        "flux_h_1fwhm_aper","flux_h_2fwhm_aper","flux_h_3fwhm_aper",
        "fluxerr_h_1fwhm_aper", "fluxerr_h_2fwhm_aper", "fluxerr_h_3fwhm_aper",
        
        "flux_vis_sersic", "flux_y_sersic", "flux_j_sersic", "flux_h_sersic",
        "fluxerr_vis_sersic", "fluxerr_y_sersic", "fluxerr_j_sersic", "fluxerr_h_sersic"
           ]
    #
    
    table = list(zip(*(result[c] for c in cols))) # [(ra,dec,fy1,fy3,fj1,fj3,fh1,fh3), ...]
    #table
    
    #EUCLID KEYWORD HERE
    ra = []
    dec = []
    vis1_flux = []
    vis2_flux = []
    vis3_flux = []
    
    y1_flux = []
    y2_flux = []
    y3_flux = []

    vis1_fluxerr = []
    vis2_fluxerr = []
    vis3_fluxerr = []
    
    y1_fluxerr = []
    y2_fluxerr = []
    y3_fluxerr = []

    j1_flux = []
    j2_flux = []
    j3_flux = []

    j1_fluxerr = []
    j2_fluxerr = []
    j3_fluxerr = []

    h1_flux = []
    h2_flux = []
    h3_flux = []

    h1_fluxerr = []
    h2_fluxerr = []
    h3_fluxerr = []

    viss_flux  = []
    viss_fluxerr  = []
    ys_flux = []
    ys_fluxerr = []
    js_flux = []
    js_fluxerr = []
    hs_flux = []
    hs_fluxerr = []
    #
    
    table = np.asarray(table)
    
    i = 0
    while i < len(table):
        #EUCLID KEYWORD HERE
        ra.append(float(table[i][0]))
        dec.append(float(table[i][1]))
        vis1_flux.append(float(table[i][2]))
        vis2_flux.append(float(table[i][3]))
        vis3_flux.append(float(table[i][4]))

        vis1_fluxerr.append(float(table[i][5]))
        vis2_fluxerr.append(float(table[i][6]))
        vis3_fluxerr.append(float(table[i][7]))
        
        y1_flux.append(float(table[i][8]))
        y2_flux.append(float(table[i][9]))
        y3_flux.append(float(table[i][10]))

        y1_fluxerr.append(float(table[i][11]))
        y2_fluxerr.append(float(table[i][12]))
        y3_fluxerr.append(float(table[i][13]))

        j1_flux.append(float(table[i][14]))
        j2_flux.append(float(table[i][15]))
        j3_flux.append(float(table[i][16]))

        j1_fluxerr.append(float(table[i][17]))
        j2_fluxerr.append(float(table[i][18]))
        j3_fluxerr.append(float(table[i][19]))

        h1_flux.append(float(table[i][20]))
        h2_flux.append(float(table[i][21]))
        h3_flux.append(float(table[i][22]))

        h1_fluxerr.append(float(table[i][23]))
        h2_fluxerr.append(float(table[i][24]))
        h3_fluxerr.append(float(table[i][25]))

        viss_flux.append(float(table[i][26]))
        ys_flux.append(float(table[i][27]))
        js_flux.append(float(table[i][28]))
        hs_flux.append(float(table[i][29]))
        viss_fluxerr.append(float(table[i][30]))
        ys_fluxerr.append(float(table[i][31]))
        js_fluxerr.append(float(table[i][32]))
        hs_fluxerr.append(float(table[i][33]))
        #
        i = i + 1
        
    
    euclid_table = pd.DataFrame({
        #EUCLID KEYWORD HERE
        "ra": ra, "dec": dec,
        "flux_vis_1fwhm_aper": vis1_flux,"flux_vis_2fwhm_aper": vis2_flux,"flux_vis_3fwhm_aper": vis3_flux,
        "fluxerr_vis_1fwhm_aper": vis1_fluxerr,"fluxerr_vis_2fwhm_aper": vis2_fluxerr,"fluxerr_vis_3fwhm_aper": vis3_fluxerr,
        "flux_y_1fwhm_aper": y1_flux, "flux_y_2fwhm_aper": y2_flux, "flux_y_3fwhm_aper": y3_flux,
        "fluxerr_y_1fwhm_aper": y1_fluxerr, "fluxerr_y_2fwhm_aper": y2_fluxerr, "fluxerr_y_3fwhm_aper": y3_fluxerr, 
        "flux_j_1fwhm_aper": j1_flux, "flux_j_2fwhm_aper": j2_flux, "flux_j_3fwhm_aper": j3_flux, 
        "fluxerr_j_1fwhm_aper": j1_fluxerr, "fluxerr_j_2fwhm_aper": j2_fluxerr, "fluxerr_j_3fwhm_aper": j3_fluxerr, 
        "flux_h_1fwhm_aper": h1_flux, "flux_h_2fwhm_aper": h2_flux, "flux_h_3fwhm_aper": h3_flux,
        "fluxerr_h_1fwhm_aper": h1_fluxerr, "fluxerr_h_2fwhm_aper": h2_fluxerr, "fluxerr_h_3fwhm_aper": h3_fluxerr,
        "flux_sersic_vis": viss_flux,
        "flux_sersic_y": ys_flux, "flux_sersic_j": js_flux, "flux_sersic_h": hs_flux,
        "fluxerr_sersic_vis": viss_fluxerr,
        "fluxerr_sersic_y": ys_fluxerr, "fluxerr_sersic_j": js_fluxerr, "fluxerr_sersic_h": hs_fluxerr
                            })
        #
    euclid_table
    
    return{
    #EUCLID KEYWORD HERE
    'ra_eucl': ra,
    'dec_eucl': dec,
    'vis1_flux': vis1_flux,
    'vis2_flux': vis2_flux,
    'vis3_flux': vis3_flux,

    'vis1_fluxerr': vis1_fluxerr,
    'vis2_fluxerr': vis2_fluxerr,
    'vis3_fluxerr': vis3_fluxerr,
        
    'y1_flux': y1_flux,
    'y2_flux': y2_flux,
    'y3_flux': y3_flux,

    'y1_fluxerr': y1_fluxerr,
    'y2_fluxerr': y2_fluxerr,
    'y3_fluxerr': y3_fluxerr,

    'j1_flux': j1_flux,
    'j2_flux': j2_flux,
    'j3_flux': j3_flux,

    'j1_fluxerr': j1_fluxerr,
    'j2_fluxerr': j2_fluxerr,
    'j3_fluxerr': j3_fluxerr,

    'h1_flux': h1_flux,
    'h2_flux': h2_flux,
    'h3_flux': h3_flux,

    'h1_fluxerr': h1_fluxerr,
    'h2_fluxerr': h2_fluxerr,
    'h3_fluxerr': h3_fluxerr,

    'viss_flux': viss_flux,
    'ys_flux': ys_flux,
    'js_flux': js_flux,
    'hs_flux': hs_flux,
    'viss_fluxerr': viss_fluxerr,  
    'ys_fluxerr': ys_fluxerr,
    'js_fluxerr': js_fluxerr,
    'hs_fluxerr': hs_fluxerr,
    'euclid_table': euclid_table
    #
          }

# Catalog Crossmatch , LSST Query 2/2

Finish LSST Query. Compare LSST vs EUCLID accuracy in for radius of 1 aperture, 2 aperture, 3 aperture, and sersic profile based on spacial position (SkyCoord). Creates a combined table containing LSST data, Euclid data, and Redshift estimates.

In [ ]:
#CROSSMATCH LSST EUCLID DATA 

def crossmatchCats(lsst,eucl,rail):
    
    ra_lsst = np.asarray(lsst['ra_lsst'])
    dec_lsst = np.asarray(lsst['dec_lsst'])
    objID = np.asarray(lsst['objID'])
    
    member_ra_lsst = np.asarray(rail['member_ra'])
    member_dec_lsst = np.asarray(rail['member_dec'])
    mem_tag = np.asarray(rail['mem_tag'])
    match_z = np.asarray(rail['match_z'])
    
    lsst_cat = SkyCoord(ra_lsst*u.deg, dec_lsst*u.deg)
    lsst_member_cat = SkyCoord(member_ra_lsst*u.deg, member_dec_lsst*u.deg)
    
    ra_eucl = np.asarray(eucl['ra_eucl'])
    dec_eucl = np.asarray(eucl['dec_eucl'])
    eucl_cat = SkyCoord(ra_eucl*u.deg, dec_eucl*u.deg)
    
    vis1_flux = np.asarray(eucl['vis1_flux'])
    vis2_flux = np.asarray(eucl['vis2_flux'])
    vis3_flux = np.asarray(eucl['vis3_flux'])

    y1_flux = np.asarray(eucl['y1_flux'])
    y2_flux = np.asarray(eucl['y2_flux'])
    y3_flux = np.asarray(eucl['y3_flux'])

    j1_flux = np.asarray(eucl['j1_flux'])
    j2_flux = np.asarray(eucl['j2_flux'])
    j3_flux = np.asarray(eucl['j3_flux'])

    h1_flux = np.asarray(eucl['h1_flux'])
    h2_flux = np.asarray(eucl['h2_flux'])
    h3_flux = np.asarray(eucl['h3_flux'])

    viss_flux = np.asarray(eucl['viss_flux'])
    ys_flux = np.asarray(eucl['ys_flux'])
    js_flux = np.asarray(eucl['js_flux'])
    hs_flux = np.asarray(eucl['hs_flux'])

    vis1_fluxerr = np.asarray(eucl['vis1_fluxerr'])
    vis2_fluxerr = np.asarray(eucl['vis2_fluxerr'])
    vis3_fluxerr = np.asarray(eucl['vis3_fluxerr'])
    y1_fluxerr = np.asarray(eucl['y1_fluxerr'])
    y2_fluxerr = np.asarray(eucl['y2_fluxerr'])
    y3_fluxerr = np.asarray(eucl['y3_fluxerr'])

    j1_fluxerr = np.asarray(eucl['j1_fluxerr'])
    j2_fluxerr = np.asarray(eucl['j2_fluxerr'])
    j3_fluxerr = np.asarray(eucl['j3_fluxerr'])

    h1_fluxerr = np.asarray(eucl['h1_fluxerr'])
    h2_fluxerr = np.asarray(eucl['h2_fluxerr'])
    h3_fluxerr = np.asarray(eucl['h3_fluxerr'])

    viss_fluxerr = np.asarray(eucl['viss_fluxerr'])
    ys_fluxerr = np.asarray(eucl['ys_fluxerr'])
    js_fluxerr = np.asarray(eucl['js_fluxerr'])
    hs_fluxerr = np.asarray(eucl['hs_fluxerr'])

    #LSST KEYWORD HERE
    Sersic_Index = np.asarray(lsst['Sersic_Index'])
    Tract = np.asarray(lsst['Tract'])
    Patch = np.asarray(lsst['Patch'])
    Sersic_x = np.asarray(lsst['Sersic_x'])
    Sersic_y = np.asarray(lsst['Sersic_y'])
    Sersic_REff_X = np.asarray(lsst['Sersic_REff_X'])
    Sersic_REff_Y = np.asarray(lsst['Sersic_REff_Y'])
    Sersic_rho = np.asarray(lsst['Sersic_rho'])
    Sersic_ra = np.asarray(lsst['Sersic_ra'])
    Sersic_dec = np.asarray(lsst['Sersic_dec'])
    #
    uFlux = np.asarray(lsst['uFlux'])
    gFlux = np.asarray(lsst['gFlux'])
    rFlux = np.asarray(lsst['rFlux'])
    iFlux = np.asarray(lsst['iFlux'])
    zFlux = np.asarray(lsst['zFlux'])
    yFlux = np.asarray(lsst['yFlux'])
    uFluxErr = np.asarray(lsst['uFluxErr'])
    gFluxErr = np.asarray(lsst['gFluxErr'])
    rFluxErr = np.asarray(lsst['rFluxErr'])
    iFluxErr = np.asarray(lsst['iFluxErr'])
    zFluxErr = np.asarray(lsst['zFluxErr'])
    yFluxErr = np.asarray(lsst['yFluxErr'])
    
    idx, sep2d, _ = lsst_cat.match_to_catalog_sky(eucl_cat)
    
    lsst_idx_keep = []
    eucl_idx_keep = []
    object_ID = []
    ra_match, dec_match = [], []
    vis1flux, vis2flux, vis3flux, y1flux, y2flux, y3flux, j1flux, j2flux, j3flux,  h1flux, h2flux, h3flux, vissflux, ysflux, jsflux, hsflux = [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []
    vis1fluxerr, vis2fluxerr, vis3fluxerr, y1fluxerr, y2fluxerr, y3fluxerr, j1fluxerr, j2fluxerr, j3fluxerr,  h1fluxerr, h2fluxerr, h3fluxerr, vissfluxerr, ysfluxerr, jsfluxerr, hsfluxerr = [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []

    #LSST KEYWORD HERE
    sersic_index = []
    tract = []
    patch = []
    sersic_x = []
    sersic_y = []
    sersic_reff_x = []
    sersic_reff_y = []
    sersic_rho = []
    sersic_ra = []
    sersic_dec = []
    #
    
    uflux,gflux,rflux,iflux,zflux,yflux = [], [], [], [], [], []
    ufluxerr,gfluxerr,rfluxerr,ifluxerr,zfluxerr,yfluxerr = [], [], [], [], [], []
    
    for lsst_i, (s, eucl_i) in enumerate(zip(sep2d, idx)):
        if s < 1 * u.arcsec:  # 1 arcsec threshold
            lsst_idx_keep.append(lsst_i)
            eucl_idx_keep.append(eucl_i)
    
            ra_match.append(ra_eucl[eucl_i])
            dec_match.append(dec_eucl[eucl_i])
            vis1flux.append(vis1_flux[eucl_i])
            vis2flux.append(vis2_flux[eucl_i])
            vis3flux.append(vis3_flux[eucl_i])
            y1flux.append(y1_flux[eucl_i])
            y2flux.append(y2_flux[eucl_i])
            y3flux.append(y3_flux[eucl_i])

            j1flux.append(j1_flux[eucl_i])
            j2flux.append(j2_flux[eucl_i])
            j3flux.append(j3_flux[eucl_i])

            h1flux.append(h1_flux[eucl_i])
            h2flux.append(h2_flux[eucl_i])
            h3flux.append(h3_flux[eucl_i])

            vissflux.append(viss_flux[eucl_i])
            ysflux.append(ys_flux[eucl_i])
            jsflux.append(js_flux[eucl_i])
            hsflux.append(hs_flux[eucl_i])
            
            vis1fluxerr.append(vis1_fluxerr[eucl_i])
            vis2fluxerr.append(vis2_fluxerr[eucl_i])
            vis3fluxerr.append(vis3_fluxerr[eucl_i])
            
            y1fluxerr.append(y1_fluxerr[eucl_i])
            y2fluxerr.append(y2_fluxerr[eucl_i])
            y3fluxerr.append(y3_fluxerr[eucl_i])

            j1fluxerr.append(j1_fluxerr[eucl_i])
            j2fluxerr.append(j2_fluxerr[eucl_i])
            j3fluxerr.append(j3_fluxerr[eucl_i])

            h1fluxerr.append(h1_fluxerr[eucl_i])
            h2fluxerr.append(h2_fluxerr[eucl_i])
            h3fluxerr.append(h3_fluxerr[eucl_i])
            
            vissfluxerr.append(viss_fluxerr[eucl_i])
            ysfluxerr.append(ys_fluxerr[eucl_i])
            jsfluxerr.append(js_fluxerr[eucl_i])
            hsfluxerr.append(hs_fluxerr[eucl_i])
    
    for i in lsst_idx_keep: 
        #LSST KEYWORD HERE
        sersic_index.append(Sersic_Index[i])
        tract.append(Tract[i])
        patch.append(Patch[i])
        sersic_x.append(Sersic_x[i])
        sersic_y.append(Sersic_y[i])
        sersic_reff_x.append(Sersic_REff_X[i])
        sersic_reff_y.append(Sersic_REff_Y[i])
        sersic_rho.append(Sersic_rho[i])
        sersic_ra.append(Sersic_ra[i])
        sersic_dec.append(Sersic_dec[i])
        #
        uflux.append(uFlux[i])
        gflux.append(gFlux[i])
        rflux.append(rFlux[i])
        iflux.append(iFlux[i])
        zflux.append(zFlux[i])
        yflux.append(yFlux[i])
        ufluxerr.append(uFluxErr[i])
        gfluxerr.append(gFluxErr[i])
        rfluxerr.append(rFluxErr[i])
        ifluxerr.append(iFluxErr[i])
        zfluxerr.append(zFluxErr[i])
        yfluxerr.append(yFluxErr[i])
        object_ID.append(objID[i])

  #LSST KEYWORD HERE  
    data_table = list(zip(object_ID, mem_tag, ra_match, dec_match, match_z, sersic_index, tract, patch, sersic_x, sersic_y, sersic_reff_x, sersic_reff_y,sersic_rho, sersic_ra, sersic_dec, uflux, gflux, rflux, iflux, zflux, yflux, ufluxerr, gfluxerr, rfluxerr, ifluxerr, zfluxerr, yfluxerr, vis1flux, vis2flux, vis3flux, y1flux, y2flux, y3flux,  j1flux, j2flux, j3flux,  h1flux, h2flux, h3flux,  vissflux, ysflux, jsflux, hsflux, vis1fluxerr, vis2fluxerr, vis3fluxerr, y1fluxerr, y2fluxerr, y3fluxerr, j1fluxerr, j2fluxerr, j3fluxerr,  h1fluxerr, h2fluxerr, h3fluxerr, vissfluxerr, ysfluxerr, jsfluxerr, hsfluxerr))
  #
    #LSST KEYWORD HERE 
    cc_table = pd.DataFrame(data_table, columns=["LSST_objID","member_tag","RA", "DEC","Z", "sersic_index", "tract", "patch","sersic_x","sersic_y","sersic_reff_x","sersic_reff_y","sersic_rho","sersic_ra","sersic_dec","uFlux_LSST",'gFlux_LSST','rFlux_LSST','iFlux_LSST','zFlux_LSST','yFlux_LSST',
                                                 "uFluxErr_LSST",'gFluxErr_LSST','rFluxErr_LSST','iFluxErr_LSST','zFluxErr_LSST','yFluxErr_LSST',
                                                 "visFlux_EUCL_1aper", "visFlux_EUCL_2aper", "visFlux_EUCL_3aper", 
                                                 "yFlux_EUCL_1aper", "yFlux_EUCL_2aper", "yFlux_EUCL_3aper", 
                                                 "jFlux_EUCL_1aper", "jFlux_EUCL_2aper", "jFlux_EUCL_3aper", 
                                                 "hFlux_EUCL_1aper", "hFlux_EUCL_2aper", "hFlux_EUCL_3aper", 
                                                 "visFlux_sersic",
                                                 "yFlux_sersic", "jFlux_sersic", "hFlux_sersic",
                                                 "visFluxErr_EUCL_1aper", "visFluxErr_EUCL_2aper", "visFluxErr_EUCL_3aper",
                                                 "yFluxErr_EUCL_1aper", "yFluxErr_EUCL_2aper", "yFluxErr_EUCL_3aper",
                                                 "jFluxErr_EUCL_1aper", "jFluxErr_EUCL_2aper", "jFluxErr_EUCL_3aper", 
                                                 "hFluxErr_EUCL_1aper", "hFluxErr_EUCL_2aper", "hFluxErr_EUCL_3aper",
                                                 "visFluxErr_sersic",
                                                 "yFluxErr_sersic", "jFluxErr_sersic", "hFluxErr_sersic",
                                                 ])
    cc_table
    # 
    return{'cc_table': cc_table}

## Part 2

# Select Sky Regions

Select a sky region(s) to survey. runtime for 1 region is between 2-6 minutes. 

Ignore expected errors are A) DeprecationWarning: Conversion of an array with ndim > 0 to a scalar ... and B) UserWarning: Warning: converting a masked element to nan.

The user should set the ra and dec variables to their desired input, with a radius of 10.

In [ ]:
## USER INPUT: 

In [ ]:
# SKY REGION QUERY 

# SET RA, DEC, RADIUS TO DESIRED REGION:
# REGIONS INCLUDE: 

#######
# ECDFS Data
ECDFS_ra = [53.080047, 52.524831, 52.567217, 53.112415] 
ECDFS_dec = [-27.493224, -28.032273, -27.9467064, -27.494112]
ECDFS_radius = [10, 10, 10, 10]

# EUCLID Data
EUCLID_ra = [59.472782, 59.487316, 58.671383, 58.753116, 59.015645, 59.035830, 58.762615]
EUCLID_dec = [-48.995145, -49.000349, -49.254925, -49.201969, -48.652010, -48.587114, -48.246856]
EUCLID_radius = [10, 10, 10, 10, 10, 10, 10]

# NEW Data
NEW_ra = [59.487316]
NEW_dec = [-48.995145]
NEW_radius = [10]
#######

# OR

### SINGLE REGION 
#Ra = [58.753116]
#Dec = [-49.201969]
#Radius = [10]
#######

# NOW

#######
### SELECT 
ra = NEW_ra
dec = NEW_dec
radius = NEW_radius
###


# DO NOT CHANGE 
z_center = 1
z_range = 0.1
cluster_dataset = []

for i in range(len(ra)):
    print(f' Surveying Region {i+1}/{len(ra)}  Coordinates: ({ra[i]},{dec[i]})  Search Radius: {radius[i]} arc-minutes')
    lsst = surveyLSST(ra[i],dec[i],radius[i])
    rail = zRail(lsst,z_center,z_range)
    eucl = surveyEUCLID(ra[i],dec[i],radius[i])
    cm = crossmatchCats(lsst,eucl,rail)
    cc_table = cm['cc_table']
    cluster_dataset.append(cc_table)

    clear_output(wait=True)  # removes output from cell

# Compare Fluxes 

Compare LSST and Euclid fluxes on a 1:1 line. Compared fluxes are in 1, 2, 3 FWHM, and the Sersic model. Sersic consistently has best results.

In [ ]:
# COMPARE FLUXES ACROSS APERTURES 

def compareFluxs(cluster_dataset,color):
    i = 1
    for cluster in cluster_dataset:
        lsst_flux = cluster[f"{color}Flux_LSST"]
        lsst_fluxerr = cluster[f"{color}FluxErr_LSST"]
        eucl_flux_1 = cluster[f"{color}Flux_EUCL_1aper"]
        eucl_fluxerr_1 = cluster[f"{color}FluxErr_EUCL_1aper"]
        eucl_flux_2 = cluster[f"{color}Flux_EUCL_2aper"]
        eucl_fluxerr_2 = cluster[f"{color}FluxErr_EUCL_2aper"]
        eucl_flux_3 = cluster[f"{color}Flux_EUCL_3aper"]
        eucl_fluxerr_3 = cluster[f"{color}FluxErr_EUCL_3aper"]
        eucl_flux_ser = cluster[f"{color}Flux_sersic"]
        eucl_fluxerr_ser = cluster[f"{color}FluxErr_sersic"]

        # Weighted RMS (a rms value of 0 is ideal)
        err1 = np.sqrt(eucl_fluxerr_1**2 + lsst_fluxerr**2) #array of xy error
        weight1 = 1/err1**2 # array of weights
        diff1 = (lsst_flux - eucl_flux_1)**2
        rem1 = np.sqrt(np.sum(weight1*diff1) / np.sum(weight1))

        err2 = np.sqrt(eucl_fluxerr_2**2 + lsst_fluxerr**2) #array of xy error
        weight2 = 1/err2**2 # array of weights
        diff2 = (lsst_flux - eucl_flux_2)**2
        rem2 = np.sqrt(np.sum(weight2*diff2) / np.sum(weight2))

        err3 = np.sqrt(eucl_fluxerr_3**2 + lsst_fluxerr**2) #array of xy error
        weight3 = 1/err3**2 # array of weights
        diff3 = (lsst_flux - eucl_flux_3)**2
        rem3 = np.sqrt(np.sum(weight3*diff3) / np.sum(weight3))

        errser = np.sqrt(eucl_fluxerr_ser**2 + lsst_fluxerr**2) #array of xy error
        weightser = 1/errser**2 # array of weights
        diffser = (lsst_flux - eucl_flux_ser)**2
        remser = np.sqrt(np.sum(weightser*diffser) / np.sum(weightser))

        xy = np.linspace(0,400000,1000000)
        plt.figure(i)
        plt.figure(figsize = (15,15))
        
        plt.subplot(3,2,1)
        plt.plot(lsst_flux, eucl_flux_1*1000, 'r.', label = f'EUCLID-Flux (1-apr) REM: {rem1}')
        plt.plot(xy,xy,'--', color = 'black')
        plt.xlabel('LSST Flux')
        plt.ylabel('EUCL Flux')
        plt.title(f'({color}-Flux) Cluster #{i}')
        plt.legend()
#        plt.xlim(0,40)
#        plt.ylim(0,40)

        plt.subplot(3,2,2)
        plt.plot(lsst_flux, eucl_flux_2*1000, 'b.', label = f'EUCLID-Flux (2-apr) REM:{rem2}')
        plt.plot(xy,xy,'--', color = 'black')
        plt.xlabel('LSST Flux')
        plt.ylabel('EUCL Flux')
        plt.title(f'({color}-Flux) Cluster #{i}')
        plt.legend()
#        plt.xlim(0,40)
#        plt.ylim(0,40)

        plt.subplot(3,2,3)
        plt.plot(lsst_flux, eucl_flux_3*1000, 'g.', label = f'EUCLID-Flux (3-apr) REM:{rem3}')
        plt.plot(xy,xy,'--', color = 'black')
        plt.xlabel('LSST Flux')
        plt.ylabel('EUCL Flux')
        plt.title(f'({color}-Flux) Cluster #{i}')
        plt.legend()
#        plt.xlim(0,40)
#        plt.ylim(0,40)


        plt.subplot(3,2,4)
        plt.plot(lsst_flux, eucl_flux_ser*1000, '.', color = 'orange', label = f'EUCLID-Flux (sersic) REM:{remser}')
        plt.plot(xy,xy,'--', color = 'black')
        plt.xlabel('LSST Flux')
        plt.ylabel('EUCL Flux')
        plt.title(f'({color}-Flux) Cluster #{i}')
        plt.legend()
#        plt.xlim(0,40)
#        plt.ylim(0,40)

        i = i + 1    

In [ ]:
compareFluxs(cluster_dataset,'y')

Using sersic profiles will be most accurate as that graph is the closest to having a slope of 1. ie LSST and EUCLID have nearly 1:1 results 

# Export Current Data

Important to export to CSV here

In [ ]:
# EXPORT CLUSTER DATA

combined = pd.concat(cluster_dataset, ignore_index=True)
DATA.mkdir(parents=True, exist_ok=True)

combined.to_csv(DATA / "Output_A.csv", index=False)
#cluster_dataset[0]
#cluster_dataset[2]

# From here on we will only use the Sersic Fluxes from Euclid, but there is no reason to delete to other data from Euclid in the data table.  
# csv row limit is dependant on device. excel import limit is ~1,000,000 rows

In [ ]:
print(len(combined))

In [ ]:
print("Current working dir:", Path().resolve())
print("DATA path:", DATA)

# Filter Data

If you will not be filtering at all skip ahead to "Append Key Objects CSV"

 Remove rows with incomplete filter data 

In [ ]:
# FILTER A 
# optional to change output csv to a new name, keeping same name writes over existing file and saves time

#
Input = DATA / "Output_A.csv"
Output = DATA / "Output_A.csv"
#

df = pd.read_csv(Input)

# KEEP ALL ROWS WITH REAL NUMBER VALUES FOR THESE COLUMNS 
verify = ["uFlux_LSST","gFlux_LSST","rFlux_LSST","iFlux_LSST","zFlux_LSST","yFlux_LSST","visFlux_sersic","yFlux_sersic","jFlux_sersic","hFlux_sersic"]
#

mask = df[verify].notna().all(axis = 1) & ~df[verify].isin([""]).any(axis=1)

filteredDF = df[mask]

filteredDF.to_csv(Output, index = False)
rows_og = len(filteredDF)
rows_removed = len(df) - rows_og
print(f"{rows_og} complete rows written to '{Output}' ({rows_removed} rows with missing values removed)")

Remove any rows with a yFlux_LSST value of 100,000 or greater

In [ ]:
# FILTER B

#
Input = DATA /  "Output_A.csv"
Output = DATA / "Output_A.csv"
#

df = pd.read_csv(Input)

# LSST Y FLUX LIMIT
ymax = 100000
#

filteredDF = df[df["yFlux_LSST"] <= ymax]

filteredDF.to_csv(Output, index = False)
rows_og = len(filteredDF)
rows_removed = len(df) - rows_og
print(f"{rows_og} rows written to '{Output}' ({rows_removed} rows with yFlux_LSST values outside of range removed)")

Remove rows outside of desired Z range [0.140 : 0
.150]

In [ ]:
# FILTER C

#
Input = DATA / "Output_A.csv"
Output = DATA / "Output_A.csv"
#

df = pd.read_csv(Input)

# EDIT Z RANGE
zmin = 0.140
zmax = 0.150
#

filteredDF = df[df["Z"].between(zmin,zmax)]

filteredDF.to_csv(Output, index = False)
rows_og = len(filteredDF)
rows_removed = len(df) - rows_og
print(f"{rows_og} rows written to '{Output}' ({rows_removed} rows with Z values outside of range removed)")

FOR HELP ON ADDING / CHANGING DATA FILTERS VISIT:
https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html

Additional Filters

In [ ]:
# FILTER D ...

# Append Key Objects CSV

Add current filtered CSV to Key Objects CSV

In [ ]:
Input = DATA / "Output_A.csv"
Output = DATA / "Key_A.csv"

df = pd.read_csv(Input)

df.to_csv(Output, mode='a', index=False, header=not os.path.exists(Output))

rows_og = len(df)
print(f"{rows_og} rows added to '{Output}'")

Delete 'Output' CSV

In [ ]:
if os.path.exists(Input):
    os.remove(Input)
    print(f"Deleted '{Input}'")
else:
    print(f"File '{Input}' not found")

## Repeat Part 2 for all desired sky regions 

## Part 3

# Sort Key Objects CSV

Very helpful for later in the process 

In [ ]:
#
input_file = DATA / "Key_A.csv"
#

# SORT INPUT CSV BASED ON TRACTS AND PATCHES, USEFUL FOR LATER

output_file = DATA / "Key_A.csv"
df = pd.read_csv(input_file)

df.columns = df.columns.str.strip()

df_sorted = df.sort_values(by=["tract", "patch"], ascending=[True, True])
df_sorted.to_csv(output_file, index=False)

## Part A Done

## Proceed to Part B SED